# 01 — Train the frozen BEDOL core model


**Purpose.** Rebuild leakage-safe preprocessing and train the validation-locked BEDOL specification. The test period is never used for architecture or hyperparameter selection. Start with `MODE="FAST"` to verify the pipeline. Use `MODE="PAPER"` for the longer three-seed run.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT/"src"))
from bedol_repro import *

DATA_DIR, CONFIG_DIR = ROOT/"data", ROOT/"configs"
OUT = ROOT/"reproduction_outputs"
CKPT = OUT/"checkpoints"
OUT.mkdir(exist_ok=True); CKPT.mkdir(exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODE = "FAST"  # change to "PAPER" for the paper-style run
SEEDS = [11] if MODE == "FAST" else [11,29,47]
MAX_EPOCHS = 12 if MODE == "FAST" else 120
PATIENCE = 3 if MODE == "FAST" else 15
BATCH_SIZE = 128

manifest, frames = load_public_data(DATA_DIR, CONFIG_DIR/"site_manifest.csv")
splits = pd.read_csv(CONFIG_DIR/"chronological_splits.csv")
proc, meta, site_to_idx, hist_features, future_features = prepare_frames(frames, splits)

train_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"train")
val_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"val")
test_ds = SolarWindows(proc,splits,site_to_idx,hist_features,future_features,"test")
print("device:", DEVICE, "mode:", MODE)
print("windows:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# Train fresh models. No cached paper checkpoints are used.
models, validation_rows = [], []
for seed in SEEDS:
    model, best_val = train_one(
        seed=seed, variant="full",
        train_ds=train_ds, val_ds=val_ds,
        n_hist=len(hist_features), n_fut=len(future_features),
        n_sites=len(site_to_idx), device=DEVICE,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        batch_size=BATCH_SIZE,
    )
    models.append(model)
    validation_rows.append({"seed":seed,"best_validation_mse":best_val})
    torch.save({
        "seed":seed,
        "state_dict":model.state_dict(),
        "hist_features":hist_features,
        "future_features":future_features,
        "site_to_idx":site_to_idx,
        "hp":HP,
    }, CKPT/f"bedol_seed{seed}.pt")

pd.DataFrame(validation_rows).to_csv(OUT/"01_validation_scores.csv", index=False)
display(pd.DataFrame(validation_rows))


In [ ]:
# Generate fresh held-out predictions and save them for downstream notebooks.
pred = predict_ensemble(models,test_ds,proc,meta,DEVICE,batch_size=BATCH_SIZE)
pred.to_csv(OUT/"01_bedol_test_predictions.csv", index=False)

site_metrics, macro_metrics = summarize_predictions(pred, primary_only=True)
site_metrics.to_csv(OUT/"01_bedol_site_metrics.csv", index=False)
macro_metrics.to_csv(OUT/"01_bedol_macro_metrics.csv", index=False)
display(site_metrics)
display(macro_metrics)
print("Fresh outputs saved under reproduction_outputs/. Continue to Notebook 02.")
